In [ ]:
import numpy as np
from scipy.optimize import minimize
import optimize
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

In [3]:
# Little random example with 3 assets and cov marrix
mu = np.array([0.08, 0.10, 0.12])
cov = np.array([
    [0.18, 0.03, 0.04],
    [0.03, 0.12, 0.02],
    [0.04, 0.02, 0.10],
])

sr_solution = optimize.maximize_sharpe_ratio(mu, cov, risk_free=0.02)
gm_solution = optimize.maximize_geometric_mean(mu, cov)

np.set_printoptions(suppress=False, precision=6)

print("Sharpe weights:", np.round(sr_solution["weights"], 6))
print("Optimal Sharpe:", np.round(sr_solution["optimal_sharpe"], 6))
print("GM weights:", np.round(gm_solution["weights"], 6))
print("Optimal GM:", np.round(gm_solution["optimal_gm"], 6))


Sharpe weights: [0.03751  0.352587 0.609903]
Optimal Sharpe: 0.362629
GM weights: [0.       0.313963 0.686037]
Optimal GM: 0.083822


In [ ]:
# Settings
tickers = ["VWCE.DE", "IUSN.DE"]
interval = "1d"  # "1d", "1wk", or "1mo"

data = yf.download(
    tickers,
    interval=interval,
    auto_adjust=True,
    period="max",
)

timeseries = data["Close"]

[*********************100%***********************]  2 of 2 completed


In [ ]:
returns = timeseries.pct_change().dropna()

daily_mu = returns.mean()
daily_cov = returns.cov()

conversion_to_annual = {
    "1d": 252,
    "1wk": 52,
    "1mo": 12,
}[interval]
annual_mu = daily_mu * conversion_to_annual
annual_cov = daily_cov * conversion_to_annual
annual_vol = returns.std() * np.sqrt(conversion_to_annual)
annual_return = (1 + returns).prod() ** (conversion_to_annual / len(returns)) - 1

stats = pd.DataFrame(
    {
        "historical_return": annual_return,
        "arithmetic_return": annual_mu,
        "volatility": annual_vol,
    }
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

mu, cov = daily_mu.to_numpy(), daily_cov.to_numpy()

sharpe = optimize.maximize_sharpe_ratio(mu, cov)
gm = optimize.maximize_geometric_mean(mu, cov)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")
print("\nPortfolio weights:")
print(weights.round(4))
print("\nOptimal daily Sharpe:", round(sharpe["optimal_sharpe"], 6))
print("Approx. annualized Sharpe:", round(sharpe["optimal_sharpe"] * np.sqrt(conversion_to_annual), 6))
print("Optimal daily GM:", round(gm["optimal_gm"], 6))
print("Approx. annualized GM:", round((1 + gm["optimal_gm"]) ** conversion_to_annual - 1, 6))



Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
historical_return   0.1013   0.1257
arithmetic_return   0.1147   0.1314
volatility          0.1903   0.1604

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
IUSN.DE            0.0        0.0
VWCE.DE            1.0        1.0

Optimal daily Sharpe: 0.051586
Approx. annualized Sharpe: 0.818901
Optimal daily GM: 0.00047
Approx. annualized GM: 0.125793
